# 每个人选择最快通道，为什么整体变慢？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'coordinated_route_a': 10, 'coordinated_mean': 10.166666666666666, 'synchronous_last10': 14.75, 'partial_last10_mean_across_runs': 10.591725, 'partial_run_std': 0.10670082066965998}


## 1. 补全：人数守恒
逐轮检查两条通道人数非负且总和恒为30。

In [2]:
assert ((all_switch[["route_a","route_b"]]>=0).all().all())
assert (all_switch.route_a+all_switch.route_b==30).all()

## 2. 修改：所有人不改路
把更新概率设为0，检查每轮人数相同。

In [3]:
fixed=simulate_routes(5,switch_probability=0)
assert fixed.route_a.nunique()==1
print(fixed.head())

   round  route_a  route_b  mean_minutes
0      0       18       12         11.34
1      1       18       12         11.34
2      2       18       12         11.34
3      3       18       12         11.34
4      4       18       12         11.34


## 3. 迁移：随机分流
每位参与者独立以0.5概率去A，模拟1000次；和确定分配15/15比较平均总耗时。

In [4]:
rng2=np.random.default_rng(21)
a=rng2.binomial(30,.5,1000)
random_cost=np.mean([np.dot([x,30-x],travel_times([x,30-x])) for x in a])
fixed_cost=np.dot([15,15],travel_times([15,15]))
print(random_cost,fixed_cost)
assert random_cost>=fixed_cost-1

322.8370499999999 318.75
